In [1]:
from pathlib import Path

import pandas as pd

from research.statistics.market_data_processing import load_and_join
from research.statistics.micro_range.config import MicroRangeStatConfig
from research.statistics.micro_range.MicroRangeStatProcessor import (MicroRangeStatProcessor)

CONTEXT_PATH = Path(
    r"/home/marlonthewizard/Projects/Trading/MarketBehaviorEngine/feature_engine/context_builders/context_data/EURUSD_D1_context_data.parquet"
)

LIFECYCLE_PATH = Path(
    r"/home/marlonthewizard/Projects/Trading/MarketBehaviorEngine/feature_engine/confirmation_builders/feature_data/mql5_exports/EURUSD_D1_micro_range_live_for_mql5.csv"
)

OUTPUT_DIR = Path(
    r"/home/marlonthewizard/Projects/Trading/MarketResearchEngine/research/statistics/micro_range/data"
)

print("Context exists:", CONTEXT_PATH.exists())
print("Lifecycle exists:", LIFECYCLE_PATH.exists())
print("Output directory:", OUTPUT_DIR)

Context exists: True
Lifecycle exists: True
Output directory: /home/marlonthewizard/Projects/Trading/MarketResearchEngine/research/statistics/micro_range/data


In [2]:
config = MicroRangeStatConfig()

config.columns
config.events
config.outcomes

OutcomeConfig(horizons=(1, 2, 3, 5, 10, 20), excursion_horizon=20, r_levels=(0.5, 1.0, 1.5, 2.0), range_width_levels=(0.25, 0.5, 1.0, 2.0), risk_atr=1.0, intrabar_policy='conservative')

In [3]:
print(
    "Snapshot features:",
    len(config.snapshots.feature_columns),
)

print(
    "Change features:",
    len(config.snapshots.change_columns),
)

Snapshot features: 151
Change features: 39


In [4]:
market_data = load_and_join(
    context_path=CONTEXT_PATH,
    lifecycle_path=LIFECYCLE_PATH,
    columns=config.columns,
)

print("Rows:", len(market_data))
print("Columns:", len(market_data.columns))

market_data.tail()

Rows: 5884
Columns: 3073


,timestamp,open,high,low,close,bid_volume,ask_volume,atr,trend_net_change_5,trend_net_change_atr_5,...,micro_range_candidate_visible,micro_range_candidate_run_length,micro_range_confirmed_now,micro_range_active_live,micro_range_first_tradable_now,micro_range_invalidated_now,micro_range_candidate_upper,micro_range_candidate_lower,micro_range_confirmed_upper,micro_range_confirmed_lower
5879,2025-11-18 00:00:00+00:00,1.159005,1.160770,1.157150,1.158025,93337.259156,91533.109164,0.005439,-0.001145,-0.210516,...,0,0,False,False,False,False,1.16561,1.146885,NaN,NaN
5880,2025-11-19 00:00:00+00:00,1.158030,1.159730,1.151785,1.153945,99209.929178,94831.529127,0.005618,-0.009075,-1.615341,...,1,1,False,False,False,False,1.16561,1.149325,NaN,NaN
5881,2025-11-20 00:00:00+00:00,1.153940,1.155005,1.150210,1.153365,101736.309081,98956.858991,0.005559,-0.008710,-1.566766,...,1,2,False,False,False,False,1.16561,1.150210,NaN,NaN
5882,2025-11-21 00:00:00+00:00,1.153385,1.155260,1.149110,1.151580,115651.218908,111976.829029,0.005601,-0.007435,-1.327342,...,0,0,False,False,False,False,1.16561,1.149110,NaN,NaN
5883,2025-11-24 00:00:00+00:00,1.150750,1.155045,1.150205,1.152250,64685.399288,63101.479413,0.005547,-0.005775,-1.041097,...,0,0,False,False,False,False,1.16561,1.149110,NaN,NaN


In [5]:
#check that coinfirmations exist
c = config.columns

confirmation_mask = market_data[c.confirmed_now]

confirmation_rows = market_data.loc[
    confirmation_mask,
    [
        c.timestamp,
        c.confirmed_upper,
        c.confirmed_lower,
    ],
]

print("Confirmation rows:", len(confirmation_rows))

confirmation_rows.head(40)

Confirmation rows: 309


,timestamp,micro_range_confirmed_upper,micro_range_confirmed_lower
22,2003-06-04 00:00:00+00:00,1.192930,1.161870
65,2003-08-04 00:00:00+00:00,1.154820,1.113430
72,2003-08-13 00:00:00+00:00,1.142575,1.113430
117,2003-10-15 00:00:00+00:00,1.185955,1.153160
193,2004-01-29 00:00:00+00:00,1.277370,1.233310
200,2004-02-09 00:00:00+00:00,1.275940,1.234985
225,2004-03-15 00:00:00+00:00,1.245805,1.205510
260,2004-05-03 00:00:00+00:00,1.200960,1.175780
266,2004-05-11 00:00:00+00:00,1.217910,1.178790
271,2004-05-18 00:00:00+00:00,1.217910,1.177130


In [6]:
#checking that snapshot columns exist
missing_snapshot_columns = sorted(
    set(config.snapshots.feature_columns)
    - set(market_data.columns)
)

missing_change_columns = sorted(
    set(config.snapshots.change_columns)
    - set(market_data.columns)
)

print(
    "Missing snapshot columns:",
    len(missing_snapshot_columns),
)

print(
    "Missing change columns:",
    len(missing_change_columns),
)

missing_snapshot_columns[:30]

Missing snapshot columns: 0
Missing change columns: 0


[]

In [7]:
processor = MicroRangeStatProcessor(config)

result = processor.run(market_data)

print("Confirmed ranges:", len(result.confirmed_ranges))
print("Boundary events:", len(result.boundary_events))
print("Event outcomes:", len(result.event_outcomes))
print("Summary rows:", len(result.event_summary))
print("Validation rows:", len(result.validation_report))

Confirmed ranges: 309
Boundary events: 2924
Event outcomes: 5848
Summary rows: 68
Validation rows: 1


In [8]:
result.validation_report.head(50)

,severity,row,timestamp,issue
0,INFO,<NA>,NaT,no_lifecycle_issues_found


In [9]:
c = config.columns

market_data.loc[
    63:67,
    [
        c.timestamp,
        c.confirmed_now,
        c.first_tradable_now,
        c.active_live,
        c.invalidated_now,
        c.confirmed_upper,
        c.confirmed_lower,
    ],
]



,timestamp,micro_range_confirmed_now,micro_range_first_tradable_now,micro_range_active_live,micro_range_invalidated_now,micro_range_confirmed_upper,micro_range_confirmed_lower
63,2003-07-31 00:00:00+00:00,False,False,False,False,NaN,NaN
64,2003-08-01 00:00:00+00:00,False,False,False,False,NaN,NaN
65,2003-08-04 00:00:00+00:00,True,False,True,False,1.15482,1.11343
66,2003-08-05 00:00:00+00:00,False,True,False,True,NaN,NaN
67,2003-08-06 00:00:00+00:00,False,False,False,False,NaN,NaN


In [10]:
expected_first_tradable = (
    market_data[c.confirmed_now]
    .shift(1, fill_value=False)
    .astype(bool)
)

actual_first_tradable = market_data[c.first_tradable_now].astype(bool)

mismatch_mask = expected_first_tradable != actual_first_tradable

market_data.loc[
    mismatch_mask,
    [
        c.timestamp,
        c.confirmed_now,
        c.first_tradable_now,
        c.active_live,
        c.invalidated_now,
    ],
].head(20)

,timestamp,micro_range_confirmed_now,micro_range_first_tradable_now,micro_range_active_live,micro_range_invalidated_now


In [11]:
print("Confirmations:", market_data[c.confirmed_now].sum())
print("Expected first-tradable:", expected_first_tradable.sum())
print("Actual first-tradable:", actual_first_tradable.sum())
print("Mismatched rows:", mismatch_mask.sum())

Confirmations: 309
Expected first-tradable: 309
Actual first-tradable: 309
Mismatched rows: 0


In [12]:
result.validation_report["issue"].value_counts(
    dropna=False
)

result.confirmed_ranges.head()

,range_id,confirmation_idx,confirmation_timestamp,first_tradable_idx,first_tradable_timestamp,invalidation_idx,invalidation_timestamp,observation_end_idx,upper,lower,midpoint,width,confirmation_atr
0,1,22,2003-06-04 00:00:00+00:00,23,2003-06-05 00:00:00+00:00,35,2003-06-23 00:00:00+00:00,55,1.192930,1.16187,1.177400,0.031060,0.012638
1,2,65,2003-08-04 00:00:00+00:00,66,2003-08-05 00:00:00+00:00,66,2003-08-05 00:00:00+00:00,71,1.154820,1.11343,1.134125,0.041390,0.011351
2,3,72,2003-08-13 00:00:00+00:00,73,2003-08-14 00:00:00+00:00,75,2003-08-18 00:00:00+00:00,95,1.142575,1.11343,1.128003,0.029145,0.010934
3,4,117,2003-10-15 00:00:00+00:00,118,2003-10-16 00:00:00+00:00,130,2003-11-03 00:00:00+00:00,150,1.185955,1.15316,1.169558,0.032795,0.013642
4,5,193,2004-01-29 00:00:00+00:00,194,2004-01-30 00:00:00+00:00,195,2004-02-02 00:00:00+00:00,199,1.277370,1.23331,1.255340,0.044060,0.015227


In [13]:
result.confirmed_ranges.head()

result.boundary_events[
    [
        "event_id",
        "range_id",
        "event_type",
        "boundary_side",
        "decision_timestamp",
        "execution_timestamp",
        "phase",
        "bars_since_confirmation",
        "range_width_atr",
        "close_position_in_confirmed_range",
    ]
].head(30)

,event_id,range_id,event_type,boundary_side,decision_timestamp,execution_timestamp,phase,bars_since_confirmation,range_width_atr,close_position_in_confirmed_range
0,1,1,FIRST_TRADABLE,NONE,2003-06-04 00:00:00+00:00,2003-06-05 00:00:00+00:00,ACTIVE_RANGE,0,2.457755,0.084353
1,2,1,UPPER_TOUCH,UPPER,2003-06-16 00:00:00+00:00,2003-06-17 00:00:00+00:00,ACTIVE_RANGE,8,2.370575,0.642627
2,3,1,LOWER_TOUCH,LOWER,2003-06-19 00:00:00+00:00,2003-06-20 00:00:00+00:00,ACTIVE_RANGE,11,2.364563,0.276561
3,4,1,LOWER_WICK_BREAK_CLOSE_INSIDE,LOWER,2003-06-19 00:00:00+00:00,2003-06-20 00:00:00+00:00,ACTIVE_RANGE,11,2.364563,0.276561
4,5,1,CLOSE_BELOW_LOWER,LOWER,2003-06-20 00:00:00+00:00,2003-06-23 00:00:00+00:00,ACTIVE_RANGE,12,2.299730,-0.052157
5,6,1,2_CLOSES_BELOW_LOWER,LOWER,2003-06-23 00:00:00+00:00,2003-06-24 00:00:00+00:00,POST_INVALIDATION,13,2.349190,-0.222795
6,7,1,INVALIDATION_BREAK_DOWN,LOWER,2003-06-23 00:00:00+00:00,2003-06-24 00:00:00+00:00,POST_INVALIDATION,13,2.349190,-0.222795
7,8,1,3_CLOSES_BELOW_LOWER,LOWER,2003-06-24 00:00:00+00:00,2003-06-25 00:00:00+00:00,POST_INVALIDATION,14,2.384507,-0.393754
8,9,1,LOWER_BREAKOUT_RETEST_HOLD,LOWER,2003-06-25 00:00:00+00:00,2003-06-26 00:00:00+00:00,POST_INVALIDATION,15,2.389431,-0.263039
9,10,2,FIRST_TRADABLE,NONE,2003-08-04 00:00:00+00:00,2003-08-05 00:00:00+00:00,ACTIVE_RANGE,0,3.646271,0.527905


In [14]:
confirmation_snapshot_columns = [
    column
    for column in result.boundary_events.columns
    if column.startswith("confirmation__")
]

decision_snapshot_columns = [
    column
    for column in result.boundary_events.columns
    if column.startswith("decision__")
]

change_snapshot_columns = [
    column
    for column in result.boundary_events.columns
    if column.startswith("change__")
]

print(
    "Confirmation snapshot columns:",
    len(confirmation_snapshot_columns),
)

print(
    "Decision snapshot columns:",
    len(decision_snapshot_columns),
)

print(
    "Change snapshot columns:",
    len(change_snapshot_columns),
)

result.boundary_events[
    [
        "event_type",
        "decision_timestamp",
        *confirmation_snapshot_columns[:3],
        *decision_snapshot_columns[:3],
        *change_snapshot_columns[:3],
    ]
].head()

Confirmation snapshot columns: 151
Decision snapshot columns: 151
Change snapshot columns: 39


,event_type,decision_timestamp,confirmation__atr,confirmation__candle_range_atr,confirmation__body_size_atr,decision__atr,decision__candle_range_atr,decision__body_size_atr,change__trend_signed_efficiency_5,change__trendline_move_robust_atr_5,change__trend_linear_r2_5
0,FIRST_TRADABLE,2003-06-04 00:00:00+00:00,0.012638,0.90603,0.698711,0.012638,0.906030,0.698711,0.000000,0.000000,0.000000
1,UPPER_TOUCH,2003-06-16 00:00:00+00:00,0.012638,0.90603,0.698711,0.013102,0.870457,0.448776,1.180473,2.863383,-0.105882
2,LOWER_TOUCH,2003-06-19 00:00:00+00:00,0.012638,0.90603,0.698711,0.013136,1.178856,0.169006,0.103237,0.426402,0.003884
3,LOWER_WICK_BREAK_CLOSE_INSIDE,2003-06-19 00:00:00+00:00,0.012638,0.90603,0.698711,0.013136,1.178856,0.169006,0.103237,0.426402,0.003884
4,CLOSE_BELOW_LOWER,2003-06-20 00:00:00+00:00,0.012638,0.90603,0.698711,0.013506,1.356441,0.747079,-0.072334,0.236032,0.013007


In [15]:
result.event_outcomes.head()

,event_id,range_id,event_type,boundary_side,confirmation_idx,decision_idx,decision_timestamp,event_idx,event_timestamp,execution_idx,...,first_minus_2p0r_bar,plus_before_minus_2p0r,first_plus_0p25_range_width_bar,first_plus_0p5_range_width_bar,first_plus_1p0_range_width_bar,first_plus_2p0_range_width_bar,first_midpoint_touch_bar,first_upper_touch_bar,first_lower_touch_bar,ambiguous_bars
0,1,1,FIRST_TRADABLE,NONE,22,22,2003-06-04 00:00:00+00:00,23,2003-06-05 00:00:00+00:00,23,...,NaN,True,1.0,1.0,NaN,NaN,1.0,NaN,11.0,0
1,1,1,FIRST_TRADABLE,NONE,22,22,2003-06-04 00:00:00+00:00,23,2003-06-05 00:00:00+00:00,23,...,8.0,False,12.0,14.0,NaN,NaN,1.0,NaN,11.0,0
2,2,1,UPPER_TOUCH,UPPER,22,30,2003-06-16 00:00:00+00:00,30,2003-06-16 00:00:00+00:00,31,...,4.0,False,NaN,NaN,NaN,NaN,1.0,NaN,3.0,0
3,2,1,UPPER_TOUCH,UPPER,22,30,2003-06-16 00:00:00+00:00,30,2003-06-16 00:00:00+00:00,31,...,NaN,True,2.0,2.0,5.0,NaN,1.0,NaN,3.0,0
4,3,1,LOWER_TOUCH,LOWER,22,33,2003-06-19 00:00:00+00:00,33,2003-06-19 00:00:00+00:00,34,...,5.0,False,NaN,NaN,NaN,NaN,NaN,NaN,1.0,0


In [16]:
result.event_outcomes["direction"].value_counts()

result.event_summary.head(50)

,event_type,phase,direction,return_1b_atr_count,return_1b_atr_mean,return_1b_atr_median,return_1b_net_cost_x1p0_count,return_1b_net_cost_x1p0_mean,return_1b_net_cost_x1p0_median,return_1b_net_cost_x1p5_count,...,mae_atr_median,path_efficiency_count,path_efficiency_mean,path_efficiency_median,favorable_to_adverse_ratio_count,favorable_to_adverse_ratio_mean,favorable_to_adverse_ratio_median,inside_close_share_count,inside_close_share_mean,inside_close_share_median
0,2_CLOSES_ABOVE_UPPER,ACTIVE_RANGE,LONG,11,-0.434334,-0.493382,11,-0.004172,-0.003890,11,...,2.589129,11,-0.086685,-0.072450,11,2.792532e+00,0.649524,11,0.345455,0.200
1,2_CLOSES_ABOVE_UPPER,ACTIVE_RANGE,SHORT,11,0.434334,0.493382,11,0.004172,0.003890,11,...,2.529138,11,0.086685,0.072450,11,3.528727e+01,1.539590,11,0.345455,0.200
2,2_CLOSES_ABOVE_UPPER,POST_INVALIDATION,LONG,112,-0.103452,-0.111758,112,-0.000613,-0.000880,112,...,1.857478,112,-0.007707,0.014497,112,4.884584e+00,1.138985,112,0.245089,0.200
3,2_CLOSES_ABOVE_UPPER,POST_INVALIDATION,SHORT,112,0.103452,0.111758,112,0.000613,0.000880,112,...,1.802299,112,0.007707,-0.014497,112,7.974492e+00,0.881548,112,0.245089,0.200
4,2_CLOSES_BELOW_LOWER,ACTIVE_RANGE,LONG,6,-0.063853,0.025966,6,-0.001351,0.000155,6,...,0.988101,6,0.096482,0.163416,6,3.663764e+00,2.377782,6,0.433333,0.500
5,2_CLOSES_BELOW_LOWER,ACTIVE_RANGE,SHORT,6,0.063853,-0.025966,6,0.001351,-0.000155,6,...,2.228500,6,-0.096482,-0.163416,6,2.095314e+00,0.435532,6,0.433333,0.500
6,2_CLOSES_BELOW_LOWER,POST_INVALIDATION,LONG,103,0.029725,0.104586,103,0.000354,0.001205,103,...,1.532454,103,0.001842,-0.021541,103,1.574292e+12,1.007559,103,0.249029,0.150
7,2_CLOSES_BELOW_LOWER,POST_INVALIDATION,SHORT,103,-0.029725,-0.104586,103,-0.000354,-0.001205,103,...,1.862814,103,-0.001842,0.021541,103,2.277379e+12,0.992498,103,0.249029,0.150
8,3_CLOSES_ABOVE_UPPER,ACTIVE_RANGE,LONG,2,1.072601,1.072601,2,0.012382,0.012382,2,...,2.626327,2,-0.175507,-0.175507,2,1.724267e+00,1.724267,2,0.200000,0.200
9,3_CLOSES_ABOVE_UPPER,ACTIVE_RANGE,SHORT,2,-1.072601,-1.072601,2,-0.012382,-0.012382,2,...,2.875417,2,0.175507,0.175507,2,9.285831e-01,0.928583,2,0.200000,0.200


In [17]:
'''result.write(OUTPUT_DIR)

print(f"Results written to: {OUTPUT_DIR}")'''

'result.write(OUTPUT_DIR)\n\nprint(f"Results written to: {OUTPUT_DIR}")'

In [18]:
print("Confirmed ranges:", len(result.confirmed_ranges))
print("Boundary events:", len(result.boundary_events))
print("Directional outcomes:", len(result.event_outcomes))
print("Summary groups:", len(result.event_summary))

print("\nEvent types:")
display(
    result.boundary_events["event_type"]
    .value_counts(dropna=False)
    .rename_axis("event_type")
    .reset_index(name="count")
)

Confirmed ranges: 309
Boundary events: 2924
Directional outcomes: 5848
Summary groups: 68

Event types:


,event_type,count
0,FIRST_TRADABLE,309
1,UPPER_WICK_BREAK_CLOSE_INSIDE,301
2,LOWER_WICK_BREAK_CLOSE_INSIDE,284
3,UPPER_TOUCH,237
4,CLOSE_ABOVE_UPPER,225
5,LOWER_TOUCH,206
6,CLOSE_BELOW_LOWER,185
7,REENTRY_FROM_UPPER,149
8,2_CLOSES_ABOVE_UPPER,123
9,REENTRY_FROM_LOWER,115


In [19]:
outcomes_per_event = (
    result.event_outcomes
    .groupby("event_id")["direction"]
    .agg(["count", "nunique"])
)

display(outcomes_per_event.describe())

bad_events = outcomes_per_event[
    (outcomes_per_event["count"] != 2)
    | (outcomes_per_event["nunique"] != 2)
]

print("Events without exactly LONG and SHORT:", len(bad_events))
display(bad_events.head(20))

,count,nunique
count,2924.0,2924.0
mean,2.0,2.0
std,0.0,0.0
min,2.0,2.0
25%,2.0,2.0
50%,2.0,2.0
75%,2.0,2.0
max,2.0,2.0


Events without exactly LONG and SHORT: 0


,count,nunique
event_id,,


In [20]:
event_coverage = (
    result.event_outcomes
    .groupby(["event_type", "phase", "direction"], dropna=False)
    .size()
    .reset_index(name="samples")
    .sort_values("samples", ascending=False)
)

display(event_coverage)

,event_type,phase,direction,samples
25,FIRST_TRADABLE,ACTIVE_RANGE,SHORT,309
24,FIRST_TRADABLE,ACTIVE_RANGE,LONG,309
66,UPPER_WICK_BREAK_CLOSE_INSIDE,POST_INVALIDATION,LONG,236
67,UPPER_WICK_BREAK_CLOSE_INSIDE,POST_INVALIDATION,SHORT,236
47,LOWER_WICK_BREAK_CLOSE_INSIDE,POST_INVALIDATION,SHORT,216
...,...,...,...,...
37,LOWER_BREAKOUT_RETEST_HOLD,ACTIVE_RANGE,SHORT,6
8,3_CLOSES_ABOVE_UPPER,ACTIVE_RANGE,LONG,2
9,3_CLOSES_ABOVE_UPPER,ACTIVE_RANGE,SHORT,2
12,3_CLOSES_BELOW_LOWER,ACTIVE_RANGE,LONG,1


In [21]:
return_columns = [
    name
    for name in result.event_outcomes.columns
    if name.startswith("return_")
    and name.endswith("_atr")
    and "censored" not in name
]

print("Available ATR return targets:", return_columns)

Available ATR return targets: ['return_1b_atr', 'return_2b_atr', 'return_3b_atr', 'return_5b_atr', 'return_10b_atr', 'return_20b_atr']


In [23]:
PRIMARY_TARGET = "return_5b_atr"

basic_edge = (
    result.event_outcomes
    .groupby(
        ["event_type", "phase", "direction"],
        dropna=False,
    )[PRIMARY_TARGET]
    .agg(
        samples="count",
        mean_return="mean",
        median_return="median",
        win_rate=lambda values: (values > 0).mean(),
        standard_deviation="std",
    )
    .reset_index()
)

# Avoid treating extremely small groups as reliable discoveries.
basic_edge["enough_samples"] = basic_edge["samples"] >= 30

display(
    basic_edge[
        basic_edge["enough_samples"]
    ]
    .sort_values("mean_return", ascending=False)
    .head(30)
)

,event_type,phase,direction,samples,mean_return,median_return,win_rate,standard_deviation,enough_samples
17,CLOSE_ABOVE_UPPER,ACTIVE_RANGE,SHORT,36,0.421144,0.552746,0.722222,1.413234,True
65,UPPER_WICK_BREAK_CLOSE_INSIDE,ACTIVE_RANGE,SHORT,65,0.238687,0.097500,0.569231,1.430588,True
51,REENTRY_FROM_LOWER,POST_INVALIDATION,SHORT,107,0.199404,0.215499,0.551402,1.422772,True
30,INVALIDATION_INSIDE_DECAY,POST_INVALIDATION,LONG,108,0.172139,0.263025,0.574074,1.322399,True
59,UPPER_BREAKOUT_RETEST_HOLD,POST_INVALIDATION,SHORT,84,0.126708,0.154038,0.559524,1.388257,True
47,LOWER_WICK_BREAK_CLOSE_INSIDE,POST_INVALIDATION,SHORT,216,0.116488,0.074590,0.513889,1.538671,True
43,LOWER_TOUCH,POST_INVALIDATION,SHORT,136,0.114328,0.049995,0.510949,1.417412,True
24,FIRST_TRADABLE,ACTIVE_RANGE,LONG,309,0.110076,0.263479,0.566343,1.413227,True
39,LOWER_BREAKOUT_RETEST_HOLD,POST_INVALIDATION,SHORT,70,0.090980,0.107984,0.600000,1.215318,True
26,INVALIDATION_BREAK_DOWN,POST_INVALIDATION,LONG,71,0.084403,0.074638,0.535211,1.463781,True


In [24]:
direction_comparison = (
    basic_edge[
        basic_edge["enough_samples"]
    ]
    .pivot(
        index=["event_type", "phase"],
        columns="direction",
        values=[
            "samples",
            "mean_return",
            "median_return",
            "win_rate",
        ],
    )
)

direction_comparison.columns = [
    f"{metric}_{direction.lower()}"
    for metric, direction in direction_comparison.columns
]

direction_comparison = direction_comparison.reset_index()

direction_comparison["long_minus_short_mean"] = (
    direction_comparison["mean_return_long"]
    - direction_comparison["mean_return_short"]
)

display(
    direction_comparison.sort_values(
        "long_minus_short_mean",
        ascending=False,
    )
)

,event_type,phase,samples_long,samples_short,mean_return_long,mean_return_short,median_return_long,median_return_short,win_rate_long,win_rate_short,long_minus_short_mean
10,INVALIDATION_INSIDE_DECAY,POST_INVALIDATION,108.0,108.0,0.172139,-0.172139,0.263025,-0.263025,0.574074,0.425926,0.344278
7,FIRST_TRADABLE,ACTIVE_RANGE,309.0,309.0,0.110076,-0.110076,0.263479,-0.263479,0.566343,0.433657,0.220153
8,INVALIDATION_BREAK_DOWN,POST_INVALIDATION,71.0,71.0,0.084403,-0.084403,0.074638,-0.074638,0.535211,0.464789,0.168807
5,CLOSE_ABOVE_UPPER,POST_INVALIDATION,189.0,189.0,0.068307,-0.068307,0.069886,-0.069886,0.523810,0.476190,0.136614
12,LOWER_TOUCH,ACTIVE_RANGE,69.0,69.0,0.063075,-0.063075,-0.028748,0.028748,0.492754,0.507246,0.126149
17,REENTRY_FROM_UPPER,POST_INVALIDATION,126.0,126.0,0.053655,-0.053655,-0.079497,0.079497,0.492063,0.507937,0.107311
22,UPPER_WICK_BREAK_CLOSE_INSIDE,POST_INVALIDATION,236.0,236.0,0.051136,-0.051136,-0.071029,0.071029,0.491525,0.508475,0.102272
14,LOWER_WICK_BREAK_CLOSE_INSIDE,ACTIVE_RANGE,68.0,68.0,0.035331,-0.035331,0.187729,-0.187729,0.544118,0.455882,0.070662
9,INVALIDATION_BREAK_UP,POST_INVALIDATION,85.0,85.0,0.016707,-0.016707,0.015359,-0.015359,0.505882,0.494118,0.033414
3,3_CLOSES_BELOW_LOWER,POST_INVALIDATION,99.0,99.0,-0.006608,0.006608,-0.120374,0.120374,0.464646,0.535354,-0.013215


In [26]:
#chronological stability
outcomes = result.event_outcomes.copy()

range_dates = (
    result.confirmed_ranges[
        ["range_id", "confirmation_timestamp"]
    ]
    .drop_duplicates("range_id")
    .sort_values("confirmation_timestamp")
    .reset_index(drop=True)
)

number_of_ranges = len(range_dates)

train_end = int(number_of_ranges * 0.60)
validation_end = int(number_of_ranges * 0.80)

range_dates["split"] = "TEST"

range_dates.loc[
    :train_end - 1,
    "split",
] = "TRAIN"

range_dates.loc[
    train_end:validation_end - 1,
    "split",
] = "VALIDATION"

outcomes = outcomes.merge(
    range_dates[
        [
            "range_id",
            "confirmation_timestamp",
            "split",
        ]
    ],
    on="range_id",
    how="left",
    validate="many_to_one",
)

display(
    range_dates.groupby("split").agg(
        ranges=("range_id", "count"),
        start=("confirmation_timestamp", "min"),
        end=("confirmation_timestamp", "max"),
    )
)

print("Outcomes missing split:", outcomes["split"].isna().sum())
print("Outcomes missing confirmation date:", outcomes["confirmation_timestamp"].isna().sum())

,ranges,start,end
split,,,
TEST,62,2021-05-31 00:00:00+00:00,2025-10-20 00:00:00+00:00
TRAIN,185,2003-06-04 00:00:00+00:00,2017-02-27 00:00:00+00:00
VALIDATION,62,2017-03-07 00:00:00+00:00,2021-05-05 00:00:00+00:00


Outcomes missing split: 0
Outcomes missing confirmation date: 0


In [27]:
stability = (
    outcomes
    .groupby(
        [
            "split",
            "event_type",
            "phase",
            "direction",
        ],
        dropna=False,
    )["return_5b_atr"]
    .agg(
        samples="count",
        mean_return="mean",
        median_return="median",
        win_rate=lambda values: (values > 0).mean(),
    )
    .reset_index()
)

display(
    stability[
        stability["event_type"].isin([
            "FIRST_TRADABLE",
            "CLOSE_ABOVE_UPPER",
            "UPPER_WICK_BREAK_CLOSE_INSIDE",
            "UPPER_TOUCH",
            "LOWER_WICK_BREAK_CLOSE_INSIDE",
            "LOWER_TOUCH",
        ])
        & (stability["phase"] == "ACTIVE_RANGE")
    ].sort_values(
        ["event_type", "direction", "split"]
    )
)

,split,event_type,phase,direction,samples,mean_return,median_return,win_rate
14,TEST,CLOSE_ABOVE_UPPER,ACTIVE_RANGE,LONG,8,-0.233245,-0.343462,0.375000
82,TRAIN,CLOSE_ABOVE_UPPER,ACTIVE_RANGE,LONG,14,-0.321236,-0.485956,0.285714
144,VALIDATION,CLOSE_ABOVE_UPPER,ACTIVE_RANGE,LONG,14,-0.628423,-0.869913,0.214286
15,TEST,CLOSE_ABOVE_UPPER,ACTIVE_RANGE,SHORT,8,0.233245,0.343462,0.625000
83,TRAIN,CLOSE_ABOVE_UPPER,ACTIVE_RANGE,SHORT,14,0.321236,0.485956,0.714286
145,VALIDATION,CLOSE_ABOVE_UPPER,ACTIVE_RANGE,SHORT,14,0.628423,0.869913,0.785714
22,TEST,FIRST_TRADABLE,ACTIVE_RANGE,LONG,62,-0.074405,0.233046,0.532258
90,TRAIN,FIRST_TRADABLE,ACTIVE_RANGE,LONG,185,0.191644,0.315714,0.589189
152,VALIDATION,FIRST_TRADABLE,ACTIVE_RANGE,LONG,62,0.051171,0.078316,0.532258
23,TEST,FIRST_TRADABLE,ACTIVE_RANGE,SHORT,62,0.074405,-0.233046,0.467742


In [22]:
selected_return = return_columns[0]

basic_edge = (
    result.event_outcomes
    .groupby(["event_type", "phase", "direction"])[selected_return]
    .agg(
        samples="count",
        mean="mean",
        median="median",
        win_rate=lambda values: (values > 0).mean(),
        standard_deviation="std",
    )
    .reset_index()
    .sort_values("mean", ascending=False)
)

display(basic_edge.head(30))

,event_type,phase,direction,samples,mean,median,win_rate,standard_deviation
8,3_CLOSES_ABOVE_UPPER,ACTIVE_RANGE,LONG,2,1.072601,1.072601,1.000000,0.044308
57,UPPER_BREAKOUT_RETEST_HOLD,ACTIVE_RANGE,SHORT,10,0.477298,0.508771,0.800000,0.581057
13,3_CLOSES_BELOW_LOWER,ACTIVE_RANGE,SHORT,1,0.450347,0.450347,1.000000,NaN
1,2_CLOSES_ABOVE_UPPER,ACTIVE_RANGE,SHORT,11,0.434334,0.493382,0.818182,0.569359
17,CLOSE_ABOVE_UPPER,ACTIVE_RANGE,SHORT,36,0.302312,0.223058,0.666667,0.433199
32,INVALIDATION_WICK_DOWN,POST_INVALIDATION,LONG,22,0.208404,0.098044,0.500000,0.752614
21,CLOSE_BELOW_LOWER,ACTIVE_RANGE,SHORT,24,0.136365,0.172402,0.583333,0.550393
35,INVALIDATION_WICK_UP,POST_INVALIDATION,SHORT,23,0.127108,0.085853,0.521739,0.781191
3,2_CLOSES_ABOVE_UPPER,POST_INVALIDATION,SHORT,112,0.103452,0.111758,0.616071,0.604971
59,UPPER_BREAKOUT_RETEST_HOLD,POST_INVALIDATION,SHORT,84,0.081521,0.090687,0.547619,0.681422
